## LangGraph Open Deep Research - Supervisor-Researcher Architecture

In this notebook, we'll explore the **supervisor-researcher delegation architecture** for conducting deep research with LangGraph.

You can visit this repository to see the original application: [Open Deep Research](https://github.com/langchain-ai/open_deep_research)

Let's jump in!

## What We're Building

This implementation uses a **hierarchical delegation pattern** where:

1. **User Clarification** - Optionally asks clarifying questions to understand the research scope
2. **Research Brief Generation** - Transforms user messages into a structured research brief
3. **Supervisor** - A lead researcher that analyzes the brief and delegates research tasks
4. **Parallel Researchers** - Multiple sub-agents that conduct focused research simultaneously
5. **Research Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings are combined into a comprehensive report

![Architecture Diagram](https://i.imgur.com/Q8HEZn0.png)

This differs from a section-based approach by allowing dynamic task decomposition based on the research question, rather than predefined sections.

---

# Breakout Room #1
## Deep Research Foundations

In this breakout room, we'll understand the architecture and components of the Open Deep Research system.

## Task 1: Dependencies

You'll need API keys for Anthropic (for the LLM) and Tavily (for web search). We'll configure the system to use Anthropic's Claude Sonnet 4 exclusively.

In [ ]:
import os
import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

## Task 2: State Definitions

The state structure is hierarchical with three levels:

### Agent State (Top Level)
Contains the overall conversation messages, research brief, accumulated notes, and final report.

### Supervisor State (Middle Level)
Manages the research supervisor's messages, research iterations, and coordinating parallel researchers.

### Researcher State (Bottom Level)
Each individual researcher has their own message history, tool call iterations, and research findings.

We also have structured outputs for tool calling:
- **ConductResearch** - Tool for supervisor to delegate research to a sub-agent
- **ResearchComplete** - Tool to signal research phase is done
- **ClarifyWithUser** - Structured output for asking clarifying questions
- **ResearchQuestion** - Structured output for the research brief

Let's import these from our library: [`open_deep_library/state.py`](open_deep_library/state.py)

In [ ]:
# Import state definitions from the library
from open_deep_library.state import (
    # Main workflow states
    AgentState,           # Lines 65-72: Top-level agent state with messages, research_brief, notes, final_report
    AgentInputState,      # Lines 62-63: Input state is just messages
    
    # Supervisor states
    SupervisorState,      # Lines 74-81: Supervisor manages research delegation and iterations
    
    # Researcher states
    ResearcherState,      # Lines 83-90: Individual researcher with messages and tool iterations
    ResearcherOutputState, # Lines 92-96: Output from researcher (compressed research + raw notes)
    
    # Structured outputs for tool calling
    ConductResearch,      # Lines 15-19: Tool for delegating research to sub-agents
    ResearchComplete,     # Lines 21-22: Tool to signal research completion
    ClarifyWithUser,      # Lines 30-41: Structured output for user clarification
    ResearchQuestion,     # Lines 43-48: Structured output for research brief
)

## Task 3: Utility Functions and Tools

The system uses several key utilities:

### Search Tools
- **tavily_search** - Async web search with automatic summarization to stay within token limits
- Supports Anthropic native web search and Tavily API

### Reflection Tools
- **think_tool** - Allows researchers to reflect on their progress and plan next steps (ReAct pattern)

### Helper Utilities
- **get_all_tools** - Assembles the complete toolkit (search + MCP + reflection)
- **get_today_str** - Provides current date context for research
- Token limit handling utilities for graceful degradation

These are defined in [`open_deep_library/utils.py`](open_deep_library/utils.py)

In [ ]:
# Import utility functions and tools from the library
from open_deep_library.utils import (
    # Search tool - Lines 43-136: Tavily search with automatic summarization
    tavily_search,
    
    # Reflection tool - Lines 219-244: Strategic thinking tool for ReAct pattern
    think_tool,
    
    # Tool assembly - Lines 569-597: Get all configured tools
    get_all_tools,
    
    # Date utility - Lines 872-879: Get formatted current date
    get_today_str,
    
    # Supporting utilities for error handling
    get_api_key_for_model,          # Lines 892-914: Get API keys from config or env
    is_token_limit_exceeded,         # Lines 665-701: Detect token limit errors
    get_model_token_limit,           # Lines 831-846: Look up model's token limit
    remove_up_to_last_ai_message,    # Lines 848-866: Truncate messages for retry
    anthropic_websearch_called,      # Lines 607-637: Detect Anthropic native search usage
    openai_websearch_called,         # Lines 639-658: Detect OpenAI native search usage
    get_notes_from_tool_calls,       # Lines 599-601: Extract notes from tool messages
)

## Task 4: Configuration System

The configuration system controls:

### Research Behavior
- **allow_clarification** - Whether to ask clarifying questions before research
- **max_concurrent_research_units** - How many parallel researchers can run (default: 5)
- **max_researcher_iterations** - How many times supervisor can delegate research (default: 6)
- **max_react_tool_calls** - Tool call limit per researcher (default: 10)

### Model Configuration
- **research_model** - Model for research and supervision (we'll use Anthropic)
- **compression_model** - Model for synthesizing findings
- **final_report_model** - Model for writing the final report
- **summarization_model** - Model for summarizing web search results

### Search Configuration
- **search_api** - Which search API to use (ANTHROPIC, TAVILY, or NONE)
- **max_content_length** - Character limit before summarization

Defined in [`open_deep_library/configuration.py`](open_deep_library/configuration.py)

In [ ]:
# Import configuration from the library
from open_deep_library.configuration import (
    Configuration,    # Lines 38-247: Main configuration class with all settings
    SearchAPI,        # Lines 11-17: Enum for search API options (ANTHROPIC, TAVILY, NONE)
)

## Task 5: Prompt Templates

The system uses carefully engineered prompts for each phase:

### Phase 1: Clarification
**clarify_with_user_instructions** - Analyzes if the research scope is clear or needs clarification

### Phase 2: Research Brief
**transform_messages_into_research_topic_prompt** - Converts user messages into a detailed research brief

### Phase 3: Supervisor
**lead_researcher_prompt** - System prompt for the supervisor that manages delegation strategy

### Phase 4: Researcher
**research_system_prompt** - System prompt for individual researchers conducting focused research

### Phase 5: Compression
**compress_research_system_prompt** - Prompt for synthesizing research findings without losing information

### Phase 6: Final Report
**final_report_generation_prompt** - Comprehensive prompt for writing the final report

All prompts are defined in [`open_deep_library/prompts.py`](open_deep_library/prompts.py)

In [ ]:
# Import prompt templates from the library
from open_deep_library.prompts import (
    clarify_with_user_instructions,                    # Lines 3-41: Ask clarifying questions
    transform_messages_into_research_topic_prompt,     # Lines 44-77: Generate research brief
    lead_researcher_prompt,                            # Lines 79-136: Supervisor system prompt
    research_system_prompt,                            # Lines 138-183: Researcher system prompt
    compress_research_system_prompt,                   # Lines 186-222: Research compression prompt
    final_report_generation_prompt,                    # Lines 228-308: Final report generation
)

## Question #1:

Explain the interrelationships between the three states (Agent, Supervisor, Researcher). Why don't we just make a single huge state?

##### Answer:

The three states form a **hierarchical isolation pattern** where each level encapsulates only the information relevant to its scope:

- **AgentState** (top level) owns the overall conversation (`messages`), the `research_brief`, aggregated `notes`, and the `final_report`. It is the only state the user interacts with.
- **SupervisorState** (middle level) has its own `supervisor_messages` for planning and delegation, tracks `research_iterations`, and accumulates `notes` from completed researchers. It never sees the raw user conversation — only the research brief.
- **ResearcherState** (bottom level) contains `researcher_messages` for a single focused research thread, `tool_call_iterations` for its own loop, and outputs `compressed_research`. Each researcher is completely independent of other researchers.

**Why not a single huge state?** Three key reasons:

1. **Parallel execution safety.** Multiple researchers run concurrently. If they shared a single state, their message histories and iteration counters would collide. Separate `ResearcherState` instances mean each researcher operates in its own sandbox.
2. **Token budget management.** Each researcher's context only contains its own messages and tool outputs. If everything were in one state, every researcher would see every other researcher's messages, quickly exceeding token limits and degrading quality.
3. **Clean data flow.** The hierarchical design enforces a clear contract: researchers output `compressed_research` → supervisor aggregates `notes` → final report consumes `notes`. This prevents accidental coupling (e.g., a researcher accidentally reading another researcher's raw search results) and makes the graph easier to reason about, test, and extend.

## Question #2:

What are the advantages and disadvantages of importing these components instead of including them in the notebook?

##### Answer:

**Advantages of importing from a library:**
- **Reusability:** The same `state.py`, `prompts.py`, and `utils.py` can be used across notebooks, scripts, and production deployments without copy-pasting.
- **Maintainability:** A bug fix or prompt improvement in `prompts.py` propagates everywhere automatically. With inline code, you'd need to update every notebook independently.
- **Readability:** The notebook stays focused on the *workflow* (graph construction, configuration, execution) rather than being cluttered with 800+ lines of utility functions. This makes it easier to follow the high-level architecture.
- **Testability:** Library modules can have their own unit tests (e.g., testing `tavily_search` or `is_token_limit_exceeded` in isolation), which is awkward to do inside a notebook.

**Disadvantages of importing:**
- **Opacity:** Students and new developers can't see the implementation without opening separate files. The notebook becomes a "black box" that's harder to learn from interactively.
- **Debugging friction:** When an error occurs inside an imported function, the stack trace points into the library file, and you can't set breakpoints or add `print()` statements as easily as in a notebook cell.
- **Dependency coupling:** Changes to the library can silently break the notebook. For example, renaming a function parameter in `deep_researcher.py` would break the import without any warning in the notebook itself.
- **Environment overhead:** The library modules must be on the Python path, which adds setup complexity compared to a fully self-contained notebook.

## Activity #1: Explore the Prompts

Open `open_deep_library/prompts.py` and examine one of the prompt templates in detail.

**Requirements:**
1. Choose one prompt template (clarify, brief, supervisor, researcher, compression, or final report)
2. Explain what the prompt is designed to accomplish
3. Identify 2-3 key techniques used in the prompt (e.g., structured output, role definition, examples)
4. Suggest one improvement you might make to the prompt

### Analysis: `lead_researcher_prompt` (Supervisor System Prompt)

**What it accomplishes:** The `lead_researcher_prompt` (lines 79-136 of `prompts.py`) serves as the system prompt for the supervisor node. Its job is to turn the supervisor into a *research manager* that decomposes a research question into delegatable sub-tasks, decides how many parallel researchers to spawn, and knows when to stop delegating.

**Key techniques used:**

1. **XML-tagged sections for structured instructions.** The prompt uses `<Task>`, `<Available Tools>`, `<Instructions>`, `<Hard Limits>`, `<Show Your Thinking>`, and `<Scaling Rules>` tags to organize instructions into clearly delineated sections. This is a proven technique for Claude models — XML tags help the model parse long system prompts by providing unambiguous boundaries between instruction categories. It reduces the chance of the model conflating scaling rules with hard limits, for example.

2. **Explicit "think before you act" pattern.** The prompt enforces a ReAct-style workflow by mandating that the supervisor call `think_tool` *before* each `ConductResearch` delegation and *after* each result. This structured reflection prevents the common failure mode where agents immediately delegate without planning, leading to redundant or poorly scoped research tasks. The prompt even specifies what the thinking should cover ("Can the task be broken down?", "What's missing?").

3. **Concrete scaling heuristics with examples.** Rather than abstract instructions like "use the right number of agents," the prompt gives specific patterns: single agent for fact-finding/lists, one agent per element for comparisons. The examples ("List top 10 coffee shops → 1 agent", "Compare OpenAI vs. Anthropic vs. DeepMind → 3 agents") anchor the model's behavior in concrete scenarios, making delegation decisions more predictable.

**Suggested improvement:** The prompt currently lacks guidance on *handling contradictory findings* from different researchers. When two researchers return conflicting information (e.g., different performance figures for the same fund), the supervisor has no instruction on whether to delegate a tie-breaking research task, prefer the more authoritative source, or flag the contradiction in the notes. Adding a `<Conflict Resolution>` section — e.g., "If researchers return contradictory findings, delegate one additional focused research task to resolve the discrepancy before calling ResearchComplete" — would improve the quality of the final report on contested topics.

---

# Breakout Room #2
## Building & Running the Researcher

In this breakout room, we'll explore the node functions, build the graph, and run investment research.

## Task 6: Node Functions - The Building Blocks

Now let's look at the node functions that make up our graph. We'll import them from the library and understand what each does.

### The Complete Research Workflow

The workflow consists of 8 key nodes organized into 3 subgraphs:

1. **Main Graph Nodes:**
   - `clarify_with_user` - Entry point that checks if clarification is needed
   - `write_research_brief` - Transforms user input into structured research brief
   - `final_report_generation` - Synthesizes all research into final report

2. **Supervisor Subgraph Nodes:**
   - `supervisor` - Lead researcher that plans and delegates
   - `supervisor_tools` - Executes supervisor's tool calls (delegation, reflection)

3. **Researcher Subgraph Nodes:**
   - `researcher` - Individual researcher conducting focused research
   - `researcher_tools` - Executes researcher's tool calls (search, reflection)
   - `compress_research` - Synthesizes researcher's findings

All nodes are defined in [`open_deep_library/deep_researcher.py`](open_deep_library/deep_researcher.py)

### Node 1: clarify_with_user

**Purpose:** Analyzes user messages and asks clarifying questions if the research scope is unclear.

**Key Steps:**
1. Check if clarification is enabled in configuration
2. Use structured output to analyze if clarification is needed
3. If needed, end with a clarifying question for the user
4. If not needed, proceed to research brief with verification message

**Implementation:** [`open_deep_library/deep_researcher.py` lines 60-115](open_deep_library/deep_researcher.py#L60-L115)

In [ ]:
# Import the clarify_with_user node
from open_deep_library.deep_researcher import clarify_with_user

### Node 2: write_research_brief

**Purpose:** Transforms user messages into a structured research brief for the supervisor.

**Key Steps:**
1. Use structured output to generate detailed research brief from messages
2. Initialize supervisor with system prompt and research brief
3. Set up supervisor messages with proper context

**Why this matters:** A well-structured research brief helps the supervisor make better delegation decisions.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 118-175](open_deep_library/deep_researcher.py#L118-L175)

In [ ]:
# Import the write_research_brief node
from open_deep_library.deep_researcher import write_research_brief

### Node 3: supervisor

**Purpose:** Lead research supervisor that plans research strategy and delegates to sub-researchers.

**Key Steps:**
1. Configure model with three tools:
   - `ConductResearch` - Delegate research to a sub-agent
   - `ResearchComplete` - Signal that research is done
   - `think_tool` - Strategic reflection before decisions
2. Generate response based on current context
3. Increment research iteration count
4. Proceed to tool execution

**Decision Making:** The supervisor uses `think_tool` to reflect before delegating research, ensuring thoughtful decomposition of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 178-223](open_deep_library/deep_researcher.py#L178-L223)

In [ ]:
# Import the supervisor node (from supervisor subgraph)
from open_deep_library.deep_researcher import supervisor

### Node 4: supervisor_tools

**Purpose:** Executes the supervisor's tool calls, including strategic thinking and research delegation.

**Key Steps:**
1. Check exit conditions:
   - Exceeded maximum iterations
   - No tool calls made
   - `ResearchComplete` called
2. Process `think_tool` calls for strategic reflection
3. Execute `ConductResearch` calls in parallel:
   - Spawn researcher subgraphs for each delegation
   - Limit to `max_concurrent_research_units` (default: 5)
   - Gather all results asynchronously
4. Aggregate findings and return to supervisor

**Parallel Execution:** This is where the magic happens - multiple researchers work simultaneously on different aspects of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 225-349](open_deep_library/deep_researcher.py#L225-L349)

In [ ]:
# Import the supervisor_tools node
from open_deep_library.deep_researcher import supervisor_tools

### Node 5: researcher

**Purpose:** Individual researcher that conducts focused research on a specific topic.

**Key Steps:**
1. Load all available tools (search, MCP, reflection)
2. Configure model with tools and researcher system prompt
3. Generate response with tool calls
4. Increment tool call iteration count

**ReAct Pattern:** Researchers use `think_tool` to reflect after each search, deciding whether to continue or provide their answer.

**Available Tools:**
- Search tools (Tavily or Anthropic native search)
- `think_tool` for strategic reflection
- `ResearchComplete` to signal completion
- MCP tools (if configured)

**Implementation:** [`open_deep_library/deep_researcher.py` lines 365-424](open_deep_library/deep_researcher.py#L365-L424)

In [ ]:
# Import the researcher node (from researcher subgraph)
from open_deep_library.deep_researcher import researcher

### Node 6: researcher_tools

**Purpose:** Executes the researcher's tool calls, including searches and strategic reflection.

**Key Steps:**
1. Check early exit conditions (no tool calls, native search used)
2. Execute all tool calls in parallel:
   - Search tools fetch and summarize web content
   - `think_tool` records strategic reflections
   - MCP tools execute external integrations
3. Check late exit conditions:
   - Exceeded `max_react_tool_calls` (default: 10)
   - `ResearchComplete` called
4. Continue research loop or proceed to compression

**Error Handling:** Safely handles tool execution errors and continues with available results.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 435-509](open_deep_library/deep_researcher.py#L435-L509)

In [ ]:
# Import the researcher_tools node
from open_deep_library.deep_researcher import researcher_tools

### Node 7: compress_research

**Purpose:** Compresses and synthesizes research findings into a concise, structured summary.

**Key Steps:**
1. Configure compression model
2. Add compression instruction to messages
3. Attempt compression with retry logic:
   - If token limit exceeded, remove older messages
   - Retry up to 3 times
4. Extract raw notes from tool and AI messages
5. Return compressed research and raw notes

**Why Compression?** Researchers may accumulate lots of tool outputs and reflections. Compression ensures:
- All important information is preserved
- Redundant information is deduplicated
- Content stays within token limits for the final report

**Token Limit Handling:** Gracefully handles token limit errors by progressively truncating messages.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 511-585](open_deep_library/deep_researcher.py#L511-L585)

In [ ]:
# Import the compress_research node
from open_deep_library.deep_researcher import compress_research

### Node 8: final_report_generation

**Purpose:** Generates the final comprehensive research report from all collected findings.

**Key Steps:**
1. Extract all notes from completed research
2. Configure final report model
3. Attempt report generation with retry logic:
   - If token limit exceeded, truncate findings by 10%
   - Retry up to 3 times
4. Return final report or error message

**Token Limit Strategy:**
- First retry: Use model's token limit x 4 as character limit
- Subsequent retries: Reduce by 10% each time
- Graceful degradation with helpful error messages

**Report Quality:** The prompt guides the model to create well-structured reports with:
- Proper headings and sections
- Inline citations
- Comprehensive coverage of all findings
- Sources section at the end

**Implementation:** [`open_deep_library/deep_researcher.py` lines 607-697](open_deep_library/deep_researcher.py#L607-L697)

In [ ]:
# Import the final_report_generation node
from open_deep_library.deep_researcher import final_report_generation

## Task 7: Graph Construction - Putting It All Together

The system is organized into three interconnected graphs:

### 1. Researcher Subgraph (Bottom Level)
Handles individual focused research on a specific topic:
```
START -> researcher -> researcher_tools -> compress_research -> END
               ^            |
               +------------+ (loops until max iterations or ResearchComplete)
```

### 2. Supervisor Subgraph (Middle Level)
Manages research delegation and coordination:
```
START -> supervisor -> supervisor_tools -> END
            ^              |
            +--------------+ (loops until max iterations or ResearchComplete)
            
supervisor_tools spawns multiple researcher_subgraphs in parallel
```

### 3. Main Deep Researcher Graph (Top Level)
Orchestrates the complete research workflow:
```
START -> clarify_with_user -> write_research_brief -> research_supervisor -> final_report_generation -> END
                 |
               (may end early if clarification needed)
```

Let's import the compiled graphs from the library.

In [ ]:
# Import the pre-compiled graphs from the library
from open_deep_library.deep_researcher import (
    # Bottom level: Individual researcher workflow
    researcher_subgraph,    # Lines 588-605: researcher -> researcher_tools -> compress_research
    
    # Middle level: Supervisor coordination
    supervisor_subgraph,    # Lines 351-363: supervisor -> supervisor_tools (spawns researchers)
    
    # Top level: Complete research workflow
    deep_researcher,        # Lines 699-719: Main graph with all phases
)

## Why This Architecture?

### Advantages of Supervisor-Researcher Delegation

1. **Dynamic Task Decomposition**
   - Unlike section-based approaches with predefined structure, the supervisor can break down research based on the actual question
   - Adapts to different types of research (comparisons, lists, deep dives, etc.)

2. **Parallel Execution**
   - Multiple researchers work simultaneously on different aspects
   - Much faster than sequential section processing
   - Configurable parallelism (1-20 concurrent researchers)

3. **ReAct Pattern for Quality**
   - Researchers use `think_tool` to reflect after each search
   - Prevents excessive searching and improves search quality
   - Natural stopping conditions based on information sufficiency

4. **Flexible Tool Integration**
   - Easy to add MCP tools for specialized research
   - Supports multiple search APIs (Anthropic, Tavily)
   - Each researcher can use different tool combinations

5. **Graceful Token Limit Handling**
   - Compression prevents token overflow
   - Progressive truncation in final report generation
   - Research can scale to arbitrary depths

### Trade-offs

- **Complexity:** More moving parts than section-based approach
- **Cost:** Parallel researchers use more tokens (but faster)
- **Unpredictability:** Research structure emerges dynamically

## Task 8: Running the Deep Researcher

Now let's see the system in action! We'll use it to research investment strategies for portfolio diversification into alternatives.

### Setup

We need to:
1. Set up the investment research request
2. Configure the execution with Anthropic settings
3. Run the research workflow

In [ ]:
# Set up the graph with Anthropic configuration
from IPython.display import Markdown, display
import uuid

# Note: deep_researcher is already compiled from the library
# For this demo, we'll use it directly without additional checkpointing
graph = deep_researcher

print("\u2713 Graph ready for execution")
print("  (Note: The graph is pre-compiled from the library)")

### Configuration for Anthropic

We'll configure the system to use:
- **Claude Sonnet 4** for all research, supervision, and report generation
- **Tavily** for web search (you can also use Anthropic's native search)
- **Moderate parallelism** (1 concurrent researcher for cost control)
- **Clarification enabled** (will ask if research scope is unclear)

In [ ]:
# Configure for Anthropic with moderate settings
config = {
    "configurable": {
        # Model configuration - using Claude Sonnet 4 for everything
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,
        
        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,
        
        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,
        
        "summarization_model": "anthropic:claude-sonnet-4-20250514",
        "summarization_model_max_tokens": 8192,
        
        # Research behavior
        "allow_clarification": True,
        "max_concurrent_research_units": 1,  # 1 parallel researcher
        "max_researcher_iterations": 2,      # Supervisor can delegate up to 2 times
        "max_react_tool_calls": 3,           # Each researcher can make up to 3 tool calls
        
        # Search configuration
        "search_api": "tavily",  # Using Tavily for web search
        "max_content_length": 50000,
        
        # Thread ID for this conversation
        "thread_id": str(uuid.uuid4())
    }
}

print("\u2713 Configuration ready")
print(f"  - Research Model: Claude Sonnet 4")
print(f"  - Max Concurrent Researchers: 1")
print(f"  - Max Iterations: 2")
print(f"  - Search API: Tavily")

### Execute the Investment Research

Now let's run the research! We'll ask the system to research evidence-based strategies for incorporating alternative investments into a traditional portfolio.

The workflow will:
1. **Clarify** - Check if the request is clear (may skip if obvious)
2. **Research Brief** - Transform our request into a structured brief
3. **Supervisor** - Plan research strategy and delegate to researchers
4. **Parallel Research** - Researchers gather information simultaneously
5. **Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings combined into comprehensive report

In [ ]:
# Create our investment research request
research_request = """
I want to diversify my portfolio into alternative investments. I currently have:
- 60% equities, 30% bonds, 10% cash
- No exposure to alternatives like reinsurance, private equity, or real estate
- Limited understanding of how alternatives can reduce portfolio risk

Please research the best evidence-based strategies for incorporating alternative investments into a traditional portfolio and create a comprehensive diversification plan.
"""

# Execute the graph
async def run_research():
    """Run the research workflow and display results."""
    print("Starting research workflow...\n")
    
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_request}]},
        config,
        stream_mode="updates"
    ):
        # Display each step
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")
            
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")
            
            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")
            
            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
    
    print("\n" + "="*60)
    print("Research workflow completed!")
    print("="*60)

# Run the research
await run_research()

## Task 9: Understanding the Output

Let's break down what happened:

### Phase 1: Clarification
The system checked if your request was clear. Since you provided specific details about your portfolio, it likely proceeded without asking clarifying questions.

### Phase 2: Research Brief
Your request was transformed into a detailed research brief that guides the supervisor's delegation strategy.

### Phase 3: Supervisor Delegation
The supervisor analyzed the brief and decided how to break down the research:
- Used `think_tool` to plan strategy
- Called `ConductResearch` to delegate to researchers
- Each delegation specified a focused research topic (e.g., reinsurance strategies, private equity allocation, real estate investment trusts)

### Phase 4: Parallel Research
Researchers worked on their assigned topics:
- Each researcher used web search tools to gather information
- Used `think_tool` to reflect after each search
- Decided when they had enough information
- Compressed their findings into clean summaries

### Phase 5: Final Report
All research findings were synthesized into a comprehensive portfolio diversification plan with:
- Well-structured sections
- Evidence-based recommendations
- Practical action items
- Sources for further reading

## Task 10: Key Takeaways & Next Steps

### Architecture Benefits
1. **Dynamic Decomposition** - Research structure emerges from the question, not predefined
2. **Parallel Efficiency** - Multiple researchers work simultaneously
3. **ReAct Quality** - Strategic reflection improves search decisions
4. **Scalability** - Handles token limits gracefully through compression
5. **Flexibility** - Easy to add new tools and capabilities

### When to Use This Pattern
- **Complex research questions** that need multi-angle investigation
- **Comparison tasks** where parallel research on different topics is beneficial
- **Open-ended exploration** where structure should emerge dynamically
- **Time-sensitive research** where parallel execution speeds up results

### When to Use Section-Based Instead
- **Highly structured reports** with predefined format requirements
- **Template-based content** where sections are always the same
- **Sequential dependencies** where later sections depend on earlier ones
- **Budget constraints** where token efficiency is critical

### Extend the System
1. **Add MCP Tools** - Integrate specialized tools for your domain
2. **Custom Prompts** - Modify prompts for specific research types
3. **Different Models** - Try different Claude versions or mix models
4. **Persistence** - Use a real database for checkpointing instead of memory

### Learn More
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [Open Deep Research Repo](https://github.com/langchain-ai/open_deep_research)
- [Anthropic Claude Documentation](https://docs.anthropic.com/)
- [Tavily Search API](https://tavily.com/)

## Question #3:

What are the trade-offs of using parallel researchers vs. sequential research? When might you choose one approach over the other?

##### Answer:

**Parallel researchers** (this architecture's default) offer speed and breadth: multiple researchers investigate different facets of a question simultaneously, reducing wall-clock time roughly proportionally to the number of parallel units. For example, researching "reinsurance vs. private equity vs. real estate" can be decomposed into three independent researchers, each completing in the time of one.

**Trade-offs of parallel execution:**
- **Cost:** Each parallel researcher makes its own LLM calls and search queries, so total token usage scales linearly with the number of researchers. With `max_concurrent_research_units=5`, you could use 5x the tokens of a single sequential researcher.
- **Redundancy:** Independent researchers may search for overlapping information since they can't see each other's findings. The compression step mitigates this, but some waste is inherent.
- **Coordination difficulty:** Parallel researchers can't build on each other's discoveries. If Researcher A finds a critical insight that would change Researcher B's search strategy, B has no way to know.

**Sequential research** is better when:
- Research topics have strong dependencies (e.g., "First find the fund's NAV, then evaluate its risk-adjusted performance using that NAV").
- Budget is constrained and you want to minimize redundant searches.
- The question is narrow enough that a single researcher with more iterations is sufficient.

**Parallel research** is better when:
- The question has naturally independent sub-topics (comparisons, surveys, multi-asset analyses).
- Time-to-result matters more than token cost.
- Breadth of coverage is more important than depth on any single sub-topic.

## Question #4:

How would you adapt this deep research architecture for a production investment research application? What additional components would you need?

##### Answer:

Adapting this architecture for production investment research would require several additional components:

1. **Authentication and compliance layer:** Investment advice is regulated. You'd need user authentication, role-based access control, and compliance guardrails that prevent the system from making unsuitable recommendations (e.g., suggesting illiquid alternatives to a non-accredited investor). Every recommendation should include required disclosures.

2. **Persistent state and checkpointing:** Replace in-memory state with a durable database (e.g., PostgreSQL with LangGraph's checkpoint system). This ensures that long-running research sessions survive server restarts and that users can resume interrupted research.

3. **Proprietary data integration:** Add MCP tools or custom retrievers that access internal data sources — portfolio management systems, Bloomberg/Refinitiv terminals, internal research databases, and compliance databases. The current system only uses web search, which is insufficient for institutional-grade research.

4. **Human-in-the-loop review:** Before delivering a final report to a client, route it through a compliance officer or senior advisor for review. LangGraph's `interrupt` mechanism can pause the graph at the `final_report_generation` node and wait for human approval.

5. **Cost controls and observability:** Add per-user token budgets, rate limiting, and comprehensive logging via LangSmith or a similar platform. Track cost-per-research-session and set alerts for runaway research loops. Monitor latency, error rates, and search quality metrics.

6. **Caching and RAG integration:** Cache frequently accessed research topics to avoid redundant web searches. Integrate a vector store (like QDrant from Notebook 1) containing the firm's proprietary research library, so researchers search internal knowledge before hitting external APIs.

7. **Audit trail:** Log every search query, LLM call, tool invocation, and report generation with timestamps and user attribution for regulatory audit purposes.

## Activity #2: Custom Investment Research

Using what you've learned, run a custom investment research task.

**Requirements:**
1. Create an investment-related research question (reinsurance, private equity, real estate, commodities)
2. Modify the configuration for your use case
3. Run the research and analyze the output
4. Document what worked well and what could be improved

**Experiment ideas:**
- Research reinsurance strategies for different risk profiles
- Compare different alternative investment vehicles
- Investigate private equity fund structures and fee models
- Explore commodities as inflation hedges

**YOUR CODE HERE**

In [ ]:
# Activity #2: Custom Investment Research
# Research question: Compare reinsurance vs. commodities as portfolio diversifiers

my_investment_request = """
I'm an institutional investor evaluating two alternative asset classes for portfolio diversification:
1. Reinsurance / Insurance-Linked Securities (ILS) - specifically catastrophe bonds
2. Commodities - specifically broad commodity index funds and gold

I want to understand:
- Historical correlation of each with the S&P 500 and US Treasuries
- Risk-adjusted returns (Sharpe ratios) over the past decade
- Liquidity profiles and minimum investment requirements
- How each performs during equity market drawdowns (2008, 2020, 2022)
- Practical implementation options (funds, ETFs, direct investments)

Please create a detailed comparison report with a recommendation for a 10% alternatives allocation.
"""

# Configuration: use fewer researchers to control cost
my_config = {
    "configurable": {
        # Model configuration
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,
        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,
        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,
        "summarization_model": "anthropic:claude-sonnet-4-20250514",
        "summarization_model_max_tokens": 8192,
        
        # Research behavior - comparison task, so allow 2 parallel researchers
        "allow_clarification": False,  # Question is already detailed
        "max_concurrent_research_units": 2,  # One per asset class
        "max_researcher_iterations": 2,
        "max_react_tool_calls": 3,
        
        # Search configuration
        "search_api": "tavily",
        "max_content_length": 50000,
        
        # Thread ID
        "thread_id": str(uuid.uuid4())
    }
}

# Run the research
async def run_custom_research():
    """Run custom investment research and display results."""
    print("Starting custom investment research...")
    print(f"Topic: Reinsurance vs. Commodities as Portfolio Diversifiers\n")
    
    final_report = None
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": my_investment_request}]},
        my_config,
        stream_mode="updates"
    ):
        for node_name, node_output in event.items():
            print(f"  [{node_name}] completed")
            
            if node_name == "write_research_brief" and "research_brief" in node_output:
                print(f"    Research brief: {node_output['research_brief'][:200]}...")
            
            if node_name == "supervisor_tools" and "notes" in node_output:
                print(f"    Notes collected: {len(node_output['notes'])}")
            
            if node_name == "final_report_generation" and "final_report" in node_output:
                final_report = node_output["final_report"]
    
    if final_report:
        print("\n" + "=" * 60)
        print("FINAL RESEARCH REPORT")
        print("=" * 60)
        display(Markdown(final_report))
    
    print("\n" + "=" * 60)
    print("Research complete!")
    print("=" * 60)

await run_custom_research()

### Activity #2 Analysis

**What worked well:**
- The supervisor correctly decomposed the comparison into parallel research tasks (one for reinsurance/ILS, one for commodities)
- Setting `allow_clarification=False` avoided an unnecessary round-trip since the question was already detailed
- The compression step effectively synthesized search results into clean summaries with citations
- The final report maintained the comparison structure requested in the prompt

**What could be improved:**
- The researchers sometimes returned general information rather than the specific quantitative data requested (historical correlations, exact Sharpe ratios)
- With `max_react_tool_calls=3`, researchers may not have enough iterations to find granular performance data — increasing this to 5 for quantitative queries would help
- Adding a custom MCP tool that queries a financial data API (e.g., Bloomberg, FRED) would provide more precise and authoritative data than web search alone
- The final report could benefit from a post-processing step that validates all cited numbers against their sources